In [ ]:
import os
import re
import zstandard as zstd
import pandas as pd
import json
import io

## Extraction of the Data from Academic Torrents

In [ ]:
# define the keywords that need to be contained in the submissions and comments
keywords = [
    "gold", "gld", "xau", "gc=", "gold futures",
    "bullion", "precious metal", "au ", "auusd", "gold etf"
]

# compiling regex once
keyword_pattern = re.compile("|".join(keywords), re.IGNORECASE)

data_rows = []

# function that processes the files
def process_file(filepath, post_type):
    with open(filepath, "rb") as f:
        dctx = zstd.ZstdDecompressor(max_window_size=2**31)

        try:
            reader = dctx.stream_reader(f)
            text_stream = io.TextIOWrapper(reader, encoding="utf-8", errors="replace")

            for line in text_stream:
                line = line.strip()
                if not line:
                    continue

                try:
                    data = json.loads(line)
                except json.JSONDecodeError:
                    continue

                if data.get("subreddit") != "wallstreetbets":
                    continue

                if post_type == "submission":
                    text = (data.get("title") or "") + " " + (data.get("selftext") or "")
                    title = data.get("title")
                else:
                    text = data.get("body") or ""
                    title = ""

                if not keyword_pattern.search(text):
                    continue

                data_rows.append({
                    "post_type": post_type,
                    "id": data.get("id"),
                    "title": title,
                    "text": text,
                    "created_datetime": pd.to_datetime(
                        data.get("created_utc"), unit="s", errors="coerce"
                    )
                })

        except zstd.ZstdError as e:
            print(f"⚠️ Corruption in {filepath}: {e}")
            print("→ Keeping data processed so far, skipping rest of file.")


# MAIN LOOP
files = sorted(os.listdir("."))

# extracting all .rs files and process in order
rs_files = sorted([f for f in files if f.startswith("RS_") and f.endswith(".zst")])

for rs_file in rs_files:
    print(f"Processing {rs_file}...")

    # extracting year-month
    ym = rs_file.replace("RS_", "").replace(".zst", "")
    rc_file = f"RC_{ym}.zst"

    # processing submissions
    process_file(rs_file, "submission")

    # processing matching comments file if it exists
    if rc_file in files:
        print(f"Processing {rc_file}...")
        process_file(rc_file, "comment")

df = pd.DataFrame(data_rows)
print("Done. Total rows:", len(df))

In [3]:
import os

def detect_file_type(filepath):
    with open(filepath, "rb") as f:
        signature = f.read(8)

    if signature.startswith(b'\x28\xb5\x2f\xfd'):
        return "zstd (.zst)"
    elif signature.startswith(b'BZh'):
        return "bzip2 (.bz2)"
    
    elif signature.startswith(b'\x1f\x8b'):
        return "gzip (.gz)"
    elif signature.startswith(b'\xfd7zXZ'):
        return "xz (.xz)"
    elif signature.startswith(b'{') or signature.startswith(b'['):
        return "JSON / text"
    else:
        return "unknown / possibly corrupted"

# Iterate through all files in current directory
for filename in sorted(os.listdir(".")):
    if os.path.isfile(filename):
        try:
            ftype = detect_file_type(filename)
            print(f"{filename}: {ftype}")
        except Exception as e:
            print(f"{filename}: error reading file ({e})")

.DS_Store: unknown / possibly corrupted
RC_2021-02.zst: zstd (.zst)
RC_2021-03.zst: zstd (.zst)
RC_2021-04.zst: zstd (.zst)
RC_2021-05.zst: zstd (.zst)
RC_2021-06.zst: zstd (.zst)
RC_2021-07.zst: zstd (.zst)
RC_2021-08.zst: zstd (.zst)
RC_2021-09.zst: zstd (.zst)
RC_2021-10.zst: zstd (.zst)
RC_2021-11.zst: zstd (.zst)
RC_2021-12.zst: zstd (.zst)
RC_2022-01.zst: zstd (.zst)
RC_2022-02.zst: zstd (.zst)
RC_2022-03.zst: zstd (.zst)
RC_2022-04.zst: zstd (.zst)
RC_2022-05.zst: zstd (.zst)
RS_2021-02.zst: zstd (.zst)
RS_2021-03.zst: zstd (.zst)
RS_2021-04.zst: zstd (.zst)
RS_2021-05.zst: zstd (.zst)
RS_2021-06.zst: zstd (.zst)
RS_2021-07.zst: zstd (.zst)
RS_2021-08.zst: zstd (.zst)
RS_2021-09.zst: zstd (.zst)
RS_2021-10.zst: zstd (.zst)
RS_2021-11.zst: zstd (.zst)
RS_2021-12.zst: zstd (.zst)
RS_2022-01.zst: zstd (.zst)
RS_2022-02.zst: zstd (.zst)
RS_2022-03.zst: zstd (.zst)
RS_2022-04.zst: zstd (.zst)
RS_2022-05.zst: zstd (.zst)
read_info_2.ipynb: JSON / text
